# ApexTracking - ML Pipeline Notebook
Tento notebook dokumentuje cely proces tvorby modelu pro projekt ApexTracking:
- puvod dat (vlastni sber pres Apex API),
- cisteni a transformace dat,
- trenink 3 modelu (rank, damage/game, win rate),
- evaluace a export modelu pro produkcni nasazeni.

Poznamka k provozu aplikace: finalni aplikace se spousti na Renderu (ne pres localhost).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="whitegrid")

def pick_dataset_path() -> Path:
    candidates = [Path("data/players.csv"), Path("data/players_ready.csv")]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Nenalezen data/players.csv ani data/players_ready.csv")

def clamp_series(s: pd.Series, low: float, high: float) -> pd.Series:
    return s.clip(lower=low, upper=high)

In [ ]:
dataset_path = pick_dataset_path()
df = pd.read_csv(dataset_path)

print("Dataset:", dataset_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head(3))

In [ ]:
required_cols = [
    "level", "rank_score", "kills", "damage", "headshots",
    "games_played", "wins", "kdr", "damage_per_game", "rank"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Chybi sloupce: {missing}")

# 1) Cisteni: rank nesmi byt null
df = df.dropna(subset=["rank"]).copy()

# 2) Transformace: numericke sloupce
num_cols = ["level", "rank_score", "kills", "damage", "headshots", "games_played", "wins", "kdr", "damage_per_game"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

# 3) Filtrovani: odstraneni nevalidnich radku
df = df[df["games_played"] > 0].copy()
df = df[df["wins"] >= 0].copy()
df = df[df["wins"] <= df["games_played"]].copy()

# 4) Osetreni outlieru
df["kdr"] = clamp_series(df["kdr"], 0.0, 10.0)
df["damage_per_game"] = clamp_series(df["damage_per_game"], 0.0, 5000.0)

print("Rows po cisteni:", len(df))
print("Atributu pouzitych pro model:", 8)

In [ ]:
feature_columns = ["level", "rank_score", "kills", "damage", "headshots", "games_played", "wins", "kdr"]
X = df[feature_columns]

label_encoder = LabelEncoder()
y_rank = label_encoder.fit_transform(df["rank"].astype(str))

# Regresni target 1
y_damage = pd.to_numeric(df["damage_per_game"], errors="coerce").fillna(0.0)

# Regresni target 2
y_win_rate = (
    pd.to_numeric(df["wins"], errors="coerce").fillna(0.0)
    / pd.to_numeric(df["games_played"], errors="coerce").fillna(0.0).replace(0, 1)
).clip(lower=0.0, upper=1.0)

print("Rank classes:", sorted(df["rank"].astype(str).unique().tolist()))

In [ ]:
split_kwargs = {"test_size": 0.2, "random_state": 42}
class_counts = pd.Series(y_rank).value_counts()
can_stratify = len(class_counts) > 1 and class_counts.min() >= 2

if can_stratify:
    X_train, X_test, y_rank_train, y_rank_test, y_damage_train, y_damage_test, y_win_train, y_win_test = train_test_split(
        X, y_rank, y_damage, y_win_rate, stratify=y_rank, **split_kwargs
    )
else:
    X_train, X_test, y_rank_train, y_rank_test, y_damage_train, y_damage_test, y_win_train, y_win_test = train_test_split(
        X, y_rank, y_damage, y_win_rate, **split_kwargs
    )

rank_model = RandomForestClassifier(
    n_estimators=70,
    max_depth=12,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
 )
rank_model.fit(X_train, y_rank_train)
rank_pred = rank_model.predict(X_test)
rank_acc = accuracy_score(y_rank_test, rank_pred)

print(f"Rank accuracy: {rank_acc:.4f}")
print(classification_report(y_rank_test, rank_pred, target_names=label_encoder.classes_))

In [ ]:
damage_model = RandomForestRegressor(
    n_estimators=70,
    max_depth=12,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
 )
damage_model.fit(X_train, y_damage_train)
damage_pred = damage_model.predict(X_test)
damage_mae = mean_absolute_error(y_damage_test, damage_pred)

win_rate_model = RandomForestRegressor(
    n_estimators=70,
    max_depth=12,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
 )
win_rate_model.fit(X_train, y_win_train)
win_pred = np.clip(win_rate_model.predict(X_test), 0.0, 1.0)
win_mae = mean_absolute_error(y_win_test, win_pred)

print(f"Damage MAE: {damage_mae:.2f}")
print(f"Win rate MAE: {win_mae * 100:.2f} pp")

plt.figure(figsize=(7, 5))
plt.scatter(y_damage_test, damage_pred, alpha=0.35)
plt.xlabel("True damage/game")
plt.ylabel("Pred damage/game")
plt.title("Damage model - true vs predicted")
plt.show()

In [ ]:
bundle = {
    "rank_model": rank_model,
    "damage_model": damage_model,
    "win_rate_model": win_rate_model,
    "label_encoder": label_encoder,
    "feature_columns": feature_columns,
}

out_path = Path("model/model.pkl")
out_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, out_path, compress=("xz", 3))

size_mb = out_path.stat().st_size / (1024 * 1024)
print(f"Saved: {out_path} ({size_mb:.2f} MB)")

# Quick sanity inference
sample = X_test.iloc[[0]].copy()
rank_idx = int(rank_model.predict(sample)[0])
rank_name = label_encoder.inverse_transform(np.array([rank_idx]))[0]
print("Sample rank:", rank_name)
print("Sample damage:", float(damage_model.predict(sample)[0]))
print("Sample win rate:", float(np.clip(win_rate_model.predict(sample)[0], 0.0, 1.0)))

## Deploy and Runtime (Render only)
Finalni aplikace se spousti na Renderu, ne na localhost.

Minimalni produkcni konfigurace:
- Build Command: `pip install -r requirements.txt`
- Start Command: `python -m src.web`
- Environment variables: `DATABASE_URL`, `APEX_API_KEY`, `FLASK_SECRET_KEY`

Tento notebook slouzi jako dokumentace tvorby modelu a evaluace.